In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from feature_engine.creation import CyclicalFeatures

# Feature Engineering

Second notebook in the pipeline: **EDA → Feature Engineering → Modeling**.

We load the cleaned operated-flights data saved by the EDA, split it time-wise (train / val / test), build the **before-departure** feature set — derived time fields, cyclical (sin/cos) encodings, and a congestion count — detect target outliers on the **train** split, and save one modeling-ready dataset (with a `split` label) for the Modeling notebook.

## Load cleaned data

Read `data/processed/flights_clean.parquet` — the ~6.97 M cleaned operated flights from the EDA, with dtypes (categoricals, datetime) preserved.

In [2]:
DATA = Path("../data/processed")
flights = pd.read_parquet(DATA / "flights_clean.parquet")
print(flights.shape)
flights.head()

(6965267, 35)


,year,month,day_of_month,day_of_week,fl_date,op_unique_carrier,op_carrier_fl_num,origin,origin_city_name,origin_state_nm,...,diverted,crs_elapsed_time,actual_elapsed_time,air_time,distance,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
0,2024,1,1,1,2024-01-01,9E,4814.0,JFK,"New York, NY",New York,...,0,136.0,122.0,84.0,509.0,0,0,0,0,0
1,2024,1,1,1,2024-01-01,9E,4815.0,MSP,"Minneapolis, MN",Minnesota,...,0,130.0,114.0,88.0,622.0,0,0,0,0,0
2,2024,1,1,1,2024-01-01,9E,4817.0,JFK,"New York, NY",New York,...,0,106.0,90.0,61.0,288.0,0,0,0,0,0
3,2024,1,1,1,2024-01-01,9E,4817.0,RIC,"Richmond, VA",Virginia,...,0,111.0,76.0,51.0,288.0,0,0,0,0,0
4,2024,1,1,1,2024-01-01,9E,4818.0,DTW,"Detroit, MI",Michigan,...,0,79.0,70.0,45.0,237.0,0,0,0,0,0


## Time-aware split (70 / 10 / 20)

We sort by time (`fl_date`, then scheduled departure), then use sklearn's `train_test_split` with **`shuffle=False`** — so it slices the rows **in order**, making the split chronological: the first 70% are the earliest ~8.5 months, the last 20% the most recent ~2.4 months. Each row gets a `split` label:

- **train (70%, ~4.88 M)** — fit models and every target-based transform (encoders, outlier cap);
- **val (10%, ~0.70 M)** — compare / tune models;
- **test (20%, ~1.39 M)** — the untouched final score.

Everything fit on the target is fit on **train only**, then applied to val/test — leakage-safe by construction. (`shuffle=False` is the crucial bit: the *default* `shuffle=True` would randomize the rows and leak the future into train.)

In [3]:
flights = flights.sort_values(["fl_date", "crs_dep_time"]).reset_index(drop=True)
flights[["fl_date", "crs_dep_time"]].head()

,fl_date,crs_dep_time
0,2024-01-01,8
1,2024-01-01,9
2,2024-01-01,9
3,2024-01-01,20
4,2024-01-01,25


In [4]:

idx_train, idx_rest = train_test_split(flights.index, test_size=0.30, shuffle=False)
idx_val,   idx_test = train_test_split(idx_rest,      test_size=2 / 3, shuffle=False)

flights["split"] = "train"
flights.loc[idx_val,  "split"] = "val"
flights.loc[idx_test, "split"] = "test"
flights["split"] = flights["split"].astype("category")


In [5]:

b = flights.groupby("split", observed=True)["fl_date"].agg(["min", "max"])
assert b.loc["train", "max"] <= b.loc["val",  "min"], "a val flight precedes a train flight!"
assert b.loc["val",   "max"] <= b.loc["test", "min"], "a test flight precedes a val flight!"

b

,min,max
split,,
test,2024-10-19,2024-12-31
train,2024-01-01,2024-09-14
val,2024-09-14,2024-10-19


## Derived time features

Three schedule-time features, all known before departure:
- **`dep_hour`** — scheduled departure hour from `crs_dep_time` (hhmm); the strongest signal from the EDA.
- **`season`** — mapped from `month` (winter / spring / summer / fall).
- **`is_weekend`** — Saturday or Sunday (`day_of_week` 6 or 7).

In [6]:
flights["dep_hour"] = (flights["crs_dep_time"] // 100) % 24
flights["is_weekend"] = (flights["day_of_week"] >= 6).astype("int8")

season_map = {12: "winter", 1: "winter", 2: "winter",
              3: "spring", 4: "spring", 5: "spring",
              6: "summer", 7: "summer", 8: "summer",
              9: "fall", 10: "fall", 11: "fall"}
flights["season"] = flights["month"].map(season_map).astype("category")

flights[["crs_dep_time", "dep_hour", "day_of_week", "is_weekend", "month", "season"]].head()

,crs_dep_time,dep_hour,day_of_week,is_weekend,month,season
0,8,0,1,0,1,winter
1,9,0,1,0,1,winter
2,9,0,1,0,1,winter
3,20,0,1,0,1,winter
4,25,0,1,0,1,winter


## Cyclical encoding — feature-engine

Temporal features are **periodic** (hour 23 → hour 0, Sun → Mon), so we encode `dep_hour`, `day_of_week`, and `day_of_month` as **sin/cos** pairs with feature-engine's `CyclicalFeatures`. We pass the true periods via `max_values` (`dep_hour`→24, `day_of_week`→7, `day_of_month`→31); otherwise feature-engine uses the max *observed* value (23 for hour), which slightly distorts the cycle. `drop_original=False` keeps the raw integers (the trees use those).

We skip `month` (single year + bimodal seasonal shape — cyclical doesn't help). Note: `CyclicalFeatures` emits a **single** sin/cos pair per variable — fine for trees, but for the linear model one harmonic may underfit the ramp-shaped hour curve, so we'll add harmonics (or one-hot) if it underperforms.

### How cyclical encoding works

**The problem.** A periodic feature like hour has a wraparound: 23:00 and 00:00 are one hour apart, but as integers $|23 - 0| = 23$ — the encoding claims they're maximally *far*. One-hot removes that false distance but throws away the ordering entirely (every hour equally unrelated). We want adjacent values — *including across the wrap* — to stay close.

**The idea: put the value on a circle.** Map a value $x$ with period $P$ to an angle $\theta = \dfrac{2\pi x}{P}$, then take its position on the unit circle:

$$x_{\sin} = \sin\!\left(\frac{2\pi x}{P}\right), \qquad x_{\cos} = \cos\!\left(\frac{2\pi x}{P}\right)$$

As $x$ runs $0 \to P$, the angle runs $0 \to 2\pi$ — a full loop — so $x=0$ and $x=P$ land on the **same** point. The wraparound is built in.

**Why two columns (sin *and* cos).** A single number is ambiguous — $\sin\theta$ can't distinguish $\theta$ from $\pi-\theta$. The *pair* $(\cos\theta,\ \sin\theta)$ is a unique point on the circle, so it pins the value down. That's why each cyclic feature becomes **2** features.

**Worked example — `dep_hour`, $P = 24$:**

| hour | angle θ | (cos, sin) |
|---|---|---|
| 0 | 0 | (1.00, 0.00) |
| 6 | π/2 | (0.00, 1.00) |
| 12 | π | (−1.00, 0.00) |
| 18 | 3π/2 | (0.00, −1.00) |
| 23 | ≈ 6.02 | (0.97, −0.26) |

Hour 23 lands at (0.97, −0.26) — right next to hour 0's (1.00, 0.00); their straight-line distance is ≈ 0.27, while the raw integer gap was 23. Hour 0 vs hour 12 sit on opposite sides (distance 2) — correctly "far." Distances now match the clock.

**Why we pass `max_values` (= the period $P$).** feature-engine plugs $P =$ `max_value`, defaulting to the largest value *observed*. For hour that's 23, which squeezes 24 hours into a 23-step loop and makes 23 ≡ 0 exactly — wrong spacing. Setting $P = 24$ places the hours at their true $15°$ apart and closes the loop at the (unseen) hour 24. Same reasoning gives $P = 7$ for `day_of_week` (Sun→Mon wrap) and $P = 31$ for `day_of_month` (approximate, since months run 28–31 days).

**One harmonic vs many.** A single $(\sin,\cos)$ pair traces the circle once, so it can represent only **one** smooth hump around the cycle (one peak, one trough). Our hour→delay curve isn't a clean sine (it ramps up all day, then drops), so a linear model may need **higher harmonics** — the same transform at $2\times, 3\times$ the frequency, $\sin\!\big(\tfrac{2\pi k x}{P}\big)$ for $k = 1,2,3$ — to bend into the real shape. `CyclicalFeatures` emits only $k=1$; trees don't care (they split on the raw integer), which is why we kept the raw columns too.

In [7]:
cyclical = CyclicalFeatures(
    variables=["dep_hour", "day_of_week", "day_of_month"],
    max_values={"dep_hour": 24, "day_of_week": 7, "day_of_month": 31},
    drop_original=False,
)
flights = cyclical.fit_transform(flights)

flights[["dep_hour", "dep_hour_sin", "dep_hour_cos"]].head()

,dep_hour,dep_hour_sin,dep_hour_cos
0,0,0.0,1.0
1,0,0.0,1.0
2,0,0.0,1.0
3,0,0.0,1.0
4,0,0.0,1.0


## Congestion count

For each flight, count how many departures share its **origin, date, and scheduled hour** — a proxy for how busy the airport is at that time. From the EDA this was a *mild* positive signal. It's a **count of the schedule** (no target), so it's known before departure and safe to compute on the full frame — each date sits within one split, so there's no cross-split leakage.

In [8]:
flights["departures_in_hour"] = (
    flights.groupby(["origin", "fl_date", "dep_hour"], observed=True)["dest"]
           .transform("size")
           .astype("int32")
)
flights[["origin", "fl_date", "dep_hour", "departures_in_hour"]].head()

,origin,fl_date,dep_hour,departures_in_hour
0,SFO,2024-01-01,0,5
1,ONT,2024-01-01,0,1
2,DEN,2024-01-01,0,5
3,SFO,2024-01-01,0,5
4,ANC,2024-01-01,0,1
